In [1]:
%%capture
!pip install peft
!pip install evaluate
!pip install datasets
!pip install "transformers==4.57.2"
!pip install sentencepiece
!pip install emoji

In [2]:
import os
import re
import shutil

import emoji
import evaluate
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

In [3]:
def clean_text(text):
    if pd.isna(text):
        return text

    # 1. lowercase
    text = text.lower()

    # 2. remove @USER mentions
    text = re.sub(r"@user", "", text, flags=re.IGNORECASE)
    text = re.sub(r"@url", "", text, flags=re.IGNORECASE)

    # 3. remove URLs (actual links or placeholder "URL")
    # text = re.sub(r'http\S+|https\S+|url', '', text, flags=re.IGNORECASE)

    # 4. remove underscores, repeated underscores
    text = re.sub(r"_+", " ", text)

    # 5. remove slashes
    text = text.replace("\\", " ").replace("/", " ")

    # 6. remove emojis
    text = emoji.replace_emoji(text, replace="")

    # 7. remove quotation marks (normal + smart)
    text = re.sub(r"[\"“”]", "", text)

    # 8. normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [4]:
# Setup
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [5]:
%%time
# Load Model & Tokenizer
MODEL_NAME = "FacebookAI/xlm-roberta-large"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

base_model.to(device)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


CPU times: user 3.37 s, sys: 3.72 s, total: 7.09 s
Wall time: 6.26 s


XLMRobertaForSequenceClassification(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 1024, padding_idx=1)
      (position_embeddings): Embedding(514, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=1024, ou

In [6]:
# Load Data
DATA_DIR = "subtask1"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
DEV_DIR = os.path.join(DATA_DIR, "dev")
TEST_DIR = os.path.join(DATA_DIR, "test")


def load_split(split_dir):
    dfs = []
    if not os.path.exists(split_dir):
        print(f"Directory not found: {split_dir}")
        return pd.DataFrame()
    for file in os.listdir(split_dir):
        if file.endswith(".csv"):
            lang = file.replace(".csv", "")
            df = pd.read_csv(os.path.join(split_dir, file))
            df["lang"] = lang
            dfs.append(df)
    if not dfs:
        return pd.DataFrame()
    return pd.concat(dfs, ignore_index=True)


print("Loading Train Data...")
raw_train_df = load_split(TRAIN_DIR)
print(f"Loaded {len(raw_train_df)} training examples")

print("Loading Dev Data (Used as internal Test)...")
raw_dev_df = load_split(DEV_DIR)
print(f"Loaded {len(raw_dev_df)} dev examples")

print("Loading Test Data (For Submission)...")
raw_test_df = load_split(TEST_DIR)
print(f"Loaded {len(raw_test_df)} test examples")

Loading Train Data...
Loaded 73681 training examples
Loading Dev Data (Used as internal Test)...
Loaded 3687 dev examples
Loading Test Data (For Submission)...
Loaded 33288 test examples


In [7]:
# Preprocess Text
print("Preprocessing text (cleaning)...")
raw_train_df["text"] = raw_train_df["text"].astype(str).apply(clean_text)
raw_dev_df["text"] = raw_dev_df["text"].astype(str).apply(clean_text)
raw_test_df["text"] = raw_test_df["text"].astype(str).apply(clean_text)

Preprocessing text (cleaning)...


In [8]:
# Data Splitting
# Rename 'polarization' to 'labels'
if "polarization" in raw_train_df.columns:
    raw_train_df = raw_train_df.rename(columns={"polarization": "labels"})
if "polarization" in raw_dev_df.columns:
    raw_dev_df = raw_dev_df.rename(columns={"polarization": "labels"})

# Split Train into 95% Train / 5% Val
train_df, val_df = train_test_split(
    raw_train_df,
    test_size=0.05,
    stratify=raw_train_df["labels"],
    random_state=SEED,
    shuffle=True,
)

# Use Dev as Internal Test
test_df = raw_dev_df.copy()

print("Shape after split:")
print(f"Train:      {train_df.shape}")
print(f"Validation: {val_df.shape}")
print(f"Test (Dev): {test_df.shape}")

Shape after split:
Train:      (69996, 4)
Validation: (3685, 4)
Test (Dev): (3687, 4)


In [9]:
# Create Dataset Objects
train_dataset = Dataset.from_pandas(train_df[["text", "labels"]], preserve_index=False)
val_dataset = Dataset.from_pandas(val_df[["text", "labels"]], preserve_index=False)
test_dataset = Dataset.from_pandas(test_df[["text", "labels"]], preserve_index=False)

dataset = DatasetDict(
    {"train": train_dataset, "validation": val_dataset, "test": test_dataset}
)

In [10]:
%%time


# Tokenize
def tokenize_function(examples):
    return tokenizer(
        examples["text"], padding="max_length", truncation=True, max_length=256
    )


print("Tokenizing datasets...")
encoded_dataset = dataset.map(tokenize_function, batched=True)
encoded_dataset.set_format(
    type="torch", columns=["input_ids", "attention_mask", "labels"]
)

Tokenizing datasets...


Map:   0%|          | 0/69996 [00:00<?, ? examples/s]

Map:   0%|          | 0/3685 [00:00<?, ? examples/s]

Map:   0%|          | 0/3687 [00:00<?, ? examples/s]

CPU times: user 11.9 s, sys: 195 ms, total: 12.1 s
Wall time: 5.5 s


In [11]:
# Training Arguments
OUTPUT_DIR = "./output_results"
BATCH_SIZE = 32
EPOCHS = 50
LR = 2e-5
GRAD_ACCUM = 2

# Calculate steps
steps_per_epoch = len(encoded_dataset["train"]) // (BATCH_SIZE * GRAD_ACCUM)
eval_steps = steps_per_epoch
steps_per_epoch

1093

In [12]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_steps=1000,
    gradient_accumulation_steps=GRAD_ACCUM,
    logging_steps=eval_steps,
    eval_steps=eval_steps,
    save_steps=eval_steps * 60,  # Save less frequently to save space
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    eval_strategy="steps",
    logging_dir=f"{OUTPUT_DIR}/logs",
    report_to="none",
    fp16=torch.cuda.is_available(),
)

metric_f1 = evaluate.load("f1")
metric_acc = evaluate.load("accuracy")


def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="macro")[
        "f1"
    ]
    acc = metric_acc.compute(predictions=predictions, references=labels)["accuracy"]
    return {"f1": f1, "accuracy": acc}


trainer = Trainer(
    model=base_model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

In [13]:
%%time
# Start Training
trainer.train()

Step,Training Loss,Validation Loss,F1,Accuracy
1093,0.545100,0.458133,0.782252,0.782632
2186,0.430400,0.400407,0.811028,0.811940
3279,0.354600,0.387496,0.822053,0.822795
4372,0.289400,0.396282,0.825617,0.826052
5465,0.230200,0.485639,0.827200,0.827951
6558,0.179300,0.526021,0.825469,0.825509
7651,0.139300,0.626184,0.829264,0.829851
8744,0.114100,0.695898,0.828142,0.829579
9837,0.095100,0.744687,0.825024,0.825509
10930,0.080500,0.784791,0.827201,0.827680


CPU times: user 57min 53s, sys: 14min 38s, total: 1h 12min 31s
Wall time: 1h 12min 41s


TrainOutput(global_step=10930, training_loss=0.24579976073995383, metrics={'train_runtime': 4361.0891, 'train_samples_per_second': 802.506, 'train_steps_per_second': 12.543, 'total_flos': 3.2586843981490176e+17, 'train_loss': 0.24579976073995383, 'epoch': 9.990859232175502})

In [14]:
# Evaluation on Internal Test Set (Dev Folder)
print("Evaluating on Internal Test Set (Dev folder data)...")
preds_output = trainer.predict(encoded_dataset["test"])

pred_labels = np.argmax(preds_output.predictions, axis=1)
true_labels = preds_output.label_ids

print("\nClassification Report:")
report = classification_report(
    true_labels, pred_labels, target_names=["Not Polar (0)", "Polar (1)"], digits=4
)
print(f"\n{report}")

macro_f1 = f1_score(true_labels, pred_labels, average="macro")
print(f"Macro F1: {macro_f1:.4f}")

# Per-Language Analysis
test_df["preds"] = pred_labels
print("\n=== Macro F1 per Language ===")
results = []
for lang in sorted(test_df["lang"].unique()):
    lang_df = test_df[test_df["lang"] == lang]
    f1 = f1_score(lang_df["labels"], lang_df["preds"], average="macro")
    acc = accuracy_score(lang_df["labels"], lang_df["preds"])
    print(f"{lang}: F1={f1:.4f}, Acc={acc:.4f}, Support={len(lang_df)}")
    results.append(
        {"lang": lang, "f1_macro": f1, "accuracy": acc, "count": len(lang_df)}
    )

results_df = pd.DataFrame(results)
print(f"\nAverage Macro F1 across languages: {results_df['f1_macro'].mean():.4f}")

Evaluating on Internal Test Set (Dev folder data)...



Classification Report:

               precision    recall  f1-score   support

Not Polar (0)     0.8092    0.8073    0.8083      1744
    Polar (1)     0.8274    0.8291    0.8283      1943

     accuracy                         0.8188      3687
    macro avg     0.8183    0.8182    0.8183      3687
 weighted avg     0.8188    0.8188    0.8188      3687

Macro F1: 0.8183

=== Macro F1 per Language ===
amh: F1=0.7199, Acc=0.7771, Support=166
arb: F1=0.7673, Acc=0.7751, Support=169
ben: F1=0.8406, Acc=0.8434, Support=166
deu: F1=0.7168, Acc=0.7170, Support=159
eng: F1=0.7426, Acc=0.7688, Support=160
fas: F1=0.8740, Acc=0.8963, Support=164
hau: F1=0.6768, Acc=0.8791, Support=182
hin: F1=0.7658, Acc=0.8832, Support=137
ita: F1=0.6699, Acc=0.6747, Support=166
khm: F1=0.6756, Acc=0.9096, Support=332
mya: F1=0.8741, Acc=0.8750, Support=144
nep: F1=0.9000, Acc=0.9000, Support=100
ori: F1=0.7812, Acc=0.8305, Support=118
pan: F1=0.7694, Acc=0.7700, Support=100
pol: F1=0.7941, Acc=0.7983, Suppor

In [15]:
# Generate Submission (Test Folder)
SUBMISSION_DIR = "./subtask_1"
if os.path.exists(SUBMISSION_DIR):
    shutil.rmtree(SUBMISSION_DIR)
os.makedirs(SUBMISSION_DIR)

print("Generating predictions for submission...")

# tokenize test set for submission
submission_dataset = Dataset.from_pandas(raw_test_df[["text"]], preserve_index=False)
submission_tokenized = submission_dataset.map(tokenize_function, batched=True)
submission_tokenized.set_format(type="torch", columns=["input_ids", "attention_mask"])

# Predict
submission_preds_output = trainer.predict(submission_tokenized)
submission_labels = np.argmax(submission_preds_output.predictions, axis=1)

# Add predictions back to dataframe
raw_test_df["polarization"] = submission_labels

# Save individual files
languages = sorted(raw_test_df["lang"].unique())
print(f"Processing {len(languages)} languages for submission...")

for lang in languages:
    lang_df = raw_test_df[raw_test_df["lang"] == lang]
    output_df = lang_df[["id", "polarization"]]

    output_path = os.path.join(SUBMISSION_DIR, f"pred_{lang}.csv")
    output_df.to_csv(output_path, index=False)

print("Zipping prediction files...")
shutil.make_archive("subtask_1", "zip", SUBMISSION_DIR)
print(f"Created subtask_1.zip in {os.getcwd()}")

Generating predictions for submission...


Map:   0%|          | 0/33288 [00:00<?, ? examples/s]

Processing 22 languages for submission...
Zipping prediction files...
Created subtask_1.zip in /home/jovyan/work
